# Bangladesh Contract / Labor / Policy Vetting - Qwen3.5-9B Max-Coverage QLoRA

This notebook trains a new, uniquely named LoRA adapter for `Qwen/Qwen3.5-9B` using the existing live-source Bangladesh vetting dataset.

Goal: maximize useful dataset coverage while keeping the run practical on an A100 by using:

- 4-bit QLoRA
- memory-safe sequence length (`MAX_LEN = 1024`)
- broad curated curriculum across the full dataset
- no hard wall-clock stop
- Drive + Hugging Face checkpoint persistence

The Hugging Face docs list `Qwen/Qwen3.5-9B` as an official dense Qwen3.5 checkpoint and recommend text-generation usage for text-only generation. This notebook uses the text-only causal-LM path.

This is exploration/drafting support only, not legal advice.


## 1. Install

Qwen3.5 support is recent, so this notebook asks for a recent Transformers build. If Colab has an older cached runtime, restart once after this cell and run all again.


In [ ]:
%pip install -q -U "transformers>=5.8.1" "datasets>=2.20.0" "accelerate>=1.1.0" \
    "peft>=0.14.0" "bitsandbytes>=0.45.0" "huggingface_hub>=0.24.0" \
    "sentencepiece>=0.2.0" "protobuf>=4.25.0" "tensorboard>=2.17.0"

## 2. Authenticate

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login, whoami

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face token with write scope: ")
login(token=os.environ["HF_TOKEN"])
HF_USER = whoami(token=os.environ["HF_TOKEN"])["name"]
print("Logged in as", HF_USER)

## 3. Configuration

In [ ]:
from datetime import datetime

BASE_MODEL = "Qwen/Qwen3.5-9B"
DATA_REPO = f"{HF_USER}/bd-contract-labor-policy-vetting-live-sft"
HF_OUTPUT_REPO = f"{HF_USER}/bd-contract-labor-policy-vetting-qwen35-9b-max-coverage-lora"
RUN_NAME = "bd-contract-labor-policy-vetting-qwen35-9b-max-coverage-lora"

# Upload + Run all trains by default. Set True only for a no-cost notebook smoke test.
SKIP_TRAINING = False

# Max coverage mode: no hard wall-clock limit. MAX_STEPS=-1 lets epochs/dataset size control training.
MAX_STEPS = -1
TARGET_TRAIN_ROWS = 18000
VALIDATION_SAMPLE_ROWS = 800
MAX_LEN = 1024
MAX_CONTEXT_CHARS_FOR_TRAINING = 1500

# Memory-safe 9B QLoRA settings. Coverage stays high through TARGET_TRAIN_ROWS;
# memory is controlled with smaller micro-batches, checkpointing, and sequence length.
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 16
LEARNING_RATE = 2.0e-5
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.03
USE_GRADIENT_CHECKPOINTING = True
RESPONSE_ONLY_TRAINING = True

RUN_ID = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
BACKUP_TO_DRIVE = True
HUB_BACKUP_EVERY_SAVE = True
RESUME_FROM_DRIVE_CHECKPOINT = True
HUB_CHECKPOINT_PREFIX = "adapter-checkpoints"
FINAL_HUB_SUBFOLDER = "final-adapter"
DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/bd-contract-labor-policy-vetting"
DRIVE_BACKUP_DIR = f"{DRIVE_BACKUP_ROOT}/{RUN_NAME}"
OUTPUT_DIR = f"{DRIVE_BACKUP_DIR}/trainer-output"
LOCAL_WORK_DIR = "/content/bd_contract_labor_policy_qwen35_9b_max_coverage_lora"
FINAL_ADAPTER_DIR = f"{LOCAL_WORK_DIR}/final-adapter"
DRIVE_FINAL_ADAPTER_DIR = f"{DRIVE_BACKUP_DIR}/final-adapter"

print("dataset:", DATA_REPO)
print("base model:", BASE_MODEL)
print("adapter output:", HF_OUTPUT_REPO)
print("skip training:", SKIP_TRAINING)
print("max steps:", MAX_STEPS, "(-1 means full selected dataset/epochs)")
print("target rows:", TARGET_TRAIN_ROWS)
print("max len:", MAX_LEN)
print("micro batch:", BATCH_SIZE, "grad accum:", GRAD_ACCUM)
print("gradient checkpointing:", USE_GRADIENT_CHECKPOINTING)
print("drive backup dir:", DRIVE_BACKUP_DIR)


## 4. Persistence Preflight

This writes to both Google Drive and Hugging Face before training starts. If this fails, do not train.


In [ ]:
import json, os, time, traceback
from huggingface_hub import HfApi, create_repo, upload_file

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError("HF_TOKEN is missing. Run the authentication cell before preflight.")

api = HfApi(token=os.environ["HF_TOKEN"])

if BACKUP_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
        print("Drive backup dir:", DRIVE_BACKUP_DIR)
    except Exception:
        print("DRIVE BACKUP SETUP FAILED - refusing to train without persistent backup")
        traceback.print_exc()
        raise

def run_persistence_preflight():
    stamp = {
        "run_id": RUN_ID,
        "run_name": RUN_NAME,
        "base_model": BASE_MODEL,
        "dataset_repo": DATA_REPO,
        "adapter_repo": HF_OUTPUT_REPO,
        "drive_backup_dir": DRIVE_BACKUP_DIR,
        "output_dir": OUTPUT_DIR,
        "time": time.time(),
    }
    if BACKUP_TO_DRIVE:
        drive_probe = os.path.join(DRIVE_BACKUP_DIR, "drive-preflight.json")
        with open(drive_probe, "w", encoding="utf-8") as f:
            json.dump(stamp, f, indent=2)
        with open(drive_probe, "r", encoding="utf-8") as f:
            loaded = json.load(f)
        assert loaded["run_id"] == RUN_ID, "Drive preflight readback mismatch"
        print("Drive preflight wrote and read:", drive_probe)

    create_repo(HF_OUTPUT_REPO, repo_type="model", private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
    local_probe = "/content/hub-persistence-preflight-qwen35-9b.json"
    with open(local_probe, "w", encoding="utf-8") as f:
        json.dump(stamp, f, indent=2)
    upload_file(
        path_or_fileobj=local_probe,
        path_in_repo="persistence-probes/latest.json",
        repo_id=HF_OUTPUT_REPO,
        repo_type="model",
        token=os.environ["HF_TOKEN"],
        commit_message="Persistence preflight probe",
    )
    remote_files = set(api.list_repo_files(HF_OUTPUT_REPO, repo_type="model"))
    assert "persistence-probes/latest.json" in remote_files, "Hub preflight upload failed"
    print("Hub preflight wrote:", f"https://huggingface.co/{HF_OUTPUT_REPO}/blob/main/persistence-probes/latest.json")
    print("PERSISTENCE PREFLIGHT PASSED")

run_persistence_preflight()

## 5. Load And Curate Dataset

In [ ]:
import json, os, random
from collections import Counter
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

raw_dataset = load_dataset(DATA_REPO, token=os.environ["HF_TOKEN"])
print(raw_dataset)
print("original train tasks:", Counter(raw_dataset["train"]["task_type"]))

ALIGNMENT_DISCLAIMER = (
    "This is automated legal and business-compliance exploration support, not legal advice. "
    "Verify the cited source and consult a qualified Bangladeshi advocate or relevant professional before acting."
)

CITATION_KEYS = ["source_title", "source_url", "source_type", "source_authority", "retrieved_at", "section_id", "chunk_id"]
TRAIN_COLUMNS = ["instruction", "context", "response", "task_type", "source_title", "source_url", "topic", "refusal_reason"]

def row_blob(row):
    parts = []
    for key in ("task_type", "topic", "source_title", "context", "response"):
        value = row.get(key, "") if hasattr(row, "get") else ""
        parts.append(json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value))
    return " ".join(parts).lower()

def has_any(text, terms):
    lowered = (text or "").lower()
    return any(term.lower() in lowered for term in terms)

def is_title(row, *terms):
    return has_any(row.get("source_title", ""), terms)

def normalize_citations(value):
    if value is None:
        return []
    if isinstance(value, dict):
        if any(isinstance(v, list) for v in value.values()):
            count = max([len(v) for v in value.values() if isinstance(v, list)] or [0])
            out = []
            for idx in range(count):
                item = {key: (value.get(key)[idx] if isinstance(value.get(key), list) and idx < len(value.get(key)) else value.get(key)) for key in CITATION_KEYS}
                out.append(item)
            return out
        return [{key: value.get(key) for key in CITATION_KEYS}]
    if isinstance(value, list):
        out = []
        for item in value:
            out.extend(normalize_citations(item))
        return out
    return []

def response_json(payload):
    payload.setdefault("disclaimer", ALIGNMENT_DISCLAIMER)
    if "citations" in payload:
        payload["citations"] = normalize_citations(payload["citations"])
    return json.dumps(payload, ensure_ascii=False, indent=2)

def project_training_row(row):
    projected = {}
    for key in TRAIN_COLUMNS:
        value = row.get(key, "") if hasattr(row, "get") else ""
        if value is None:
            value = ""
        if isinstance(value, (dict, list)):
            value = json.dumps(value, ensure_ascii=False)
        projected[key] = str(value)
    return projected

LABOR_TITLES = ("labour", "labor", "epz", "bepza", "employment")
CONTRACT_TITLES = ("contract act", "sale of goods", "arbitration", "specific relief", "partnership act")
POLICY_TITLES = ("consumer", "sale of goods", "contract act", "companies", "labour", "labor")
COMPANY_CORE_TERMS = ("memorandum", "articles", "incorporation", "registered office", "subscriber", "share", "director", "register")
WEAK_COMPANY_TERMS = ("societies registration", "society", "winding", "liquidation", "charge", "debenture", "penalty", "offence", "prospectus")

def source_ok(row):
    task = row.get("task_type")
    blob = row_blob(row)
    if "negotiable instruments" in (row.get("source_title", "") or "").lower() and task not in {"source_grounded_summary", "clause_vetting", "redline_suggestion"}:
        return False
    if task in {"disciplinary_timeline_check", "general_employment_vetting"}:
        return is_title(row, *LABOR_TITLES)
    if task == "company_setup_pathway":
        return is_title(row, "companies", "????????") and has_any(blob, COMPANY_CORE_TERMS) and not has_any(blob, WEAK_COMPANY_TERMS)
    if task == "commercial_contract_vetting":
        return is_title(row, *CONTRACT_TITLES)
    if task == "company_policy_vetting":
        return is_title(row, *POLICY_TITLES)
    return True

TASK_CAPS = {
    "company_policy_vetting": 2600,
    "commercial_contract_vetting": 1700,
    "company_setup_pathway": 1100,
    "general_employment_vetting": 1300,
    "partnership_jv_vetting": 900,
    "expansion_pathway": 700,
    "compliance_checklist": 1600,
    "fact_intake_triage": 1500,
    "expert_handoff_packet": 1200,
    "benchmark_alignment": 1500,
    "clause_comparison": 1200,
    "source_grounded_summary": 1000,
    "clause_vetting": 900,
    "redline_suggestion": 900,
    "refusal": 240,
    "disciplinary_timeline_check": 180,
    "epz_applicability": 240,
    "foreign_investor_orientation": 180,
    "bilingual_term_mapping": 500,
    "clarification": 360,
}

blocks = []
for offset, (task, cap) in enumerate(TASK_CAPS.items()):
    subset = raw_dataset["train"].filter(lambda row, task=task: row.get("task_type") == task and source_ok(row), desc=f"filter {task}")
    if len(subset):
        subset = subset.shuffle(seed=100 + offset).select(range(min(cap, len(subset))))
        blocks.append(Dataset.from_list([project_training_row(dict(row)) for row in subset]))

if not blocks:
    raise RuntimeError("No training blocks selected. Check DATA_REPO and task names.")

train_base = concatenate_datasets(blocks).shuffle(seed=42)
if len(train_base) > TARGET_TRAIN_ROWS:
    train_base = train_base.select(range(TARGET_TRAIN_ROWS))

strong_company_citation = {
    "source_title": "???????? ???, ????",
    "source_url": "http://bdlaws.minlaw.gov.bd/act-print-788.html#section=6",
    "source_type": "bdlaws_act_print",
    "source_authority": "Laws of Bangladesh",
    "retrieved_at": "2026-05-17T08:27:57.909287+00:00",
    "section_id": "6",
    "chunk_id": "qwen35-9b-strong-company-setup-anchor-section-6",
}
strong_company_context = (
    "strong_company_setup_anchor\n"
    "Source excerpt from Companies Act 1994, section 6:\n"
    "Any seven or more persons, or where the company to be formed will be a private company, any two or more persons, "
    "associated for any lawful purpose may, by subscribing their names to a memorandum of association and otherwise "
    "complying with the requirements of this Act in respect of registration, form an incorporated company, with or without limited liability."
)
strong_company_response = response_json({
    "risk_level": "review_required",
    "source_supported_setup_points": [
        "The excerpt supports using memorandum/subscriber/registration requirements as the primary Companies Act setup basis.",
        "For a private company, the excerpt supports that two or more persons may subscribe to the memorandum for a lawful purpose and comply with registration requirements to form an incorporated company.",
        "Do not use weak registrar-reference, society, winding-up, charge, debenture, penalty, court, or prospectus excerpts as the primary basis for setup guidance."
    ],
    "broader_checks_requiring_additional_sources": ["RJSC forms/portal practice", "trade licence", "TIN", "BIN/VAT", "sector approvals", "foreign-investment or remittance papers"],
    "missing_facts_to_confirm": ["entity type", "subscribers/shareholders", "directors", "capital", "registered office", "business objects", "foreign ownership or EPZ/EZ status"],
    "source_grounding": strong_company_context,
    "citations": [strong_company_citation],
})
company_anchor = {
    "instruction": "A Bangladesh founder is incorporating a private limited company in Dhaka. Use the cited Companies Act setup excerpt and return JSON with citations.",
    "context": strong_company_context,
    "response": strong_company_response,
    "task_type": "company_setup_pathway",
    "source_title": strong_company_citation["source_title"],
    "source_url": strong_company_citation["source_url"],
    "topic": "company_setup",
    "refusal_reason": "",
}

alignment_rows = [company_anchor] * 260
alignment_rows += [{
    "instruction": "The user asks for a company setup pathway, but the supplied source is about a negotiable instrument. Respond safely in JSON without pretending the excerpt supports incorporation steps.",
    "context": "Source excerpt from The Negotiable Instruments Act, 1881: an agent signature/payment-liability provision.",
    "response": response_json({
        "answer": "I cannot ground a Bangladesh company setup pathway in this excerpt.",
        "source_limit": "The cited excerpt is not an incorporation, RJSC, memorandum/articles, trade licence, tax, or company-setup source.",
        "safe_alternative": "Retrieve Companies Act/RJSC and related sources before giving setup steps.",
        "citations": [],
    }),
    "task_type": "refusal",
    "source_title": "The Negotiable Instruments Act, 1881",
    "source_url": "http://bdlaws.minlaw.gov.bd/act-print-46.html#section=28",
    "topic": "source_mismatch",
    "refusal_reason": "source_does_not_support_requested_task",
}] * 160

alignment_dataset = Dataset.from_list([project_training_row(row) for row in alignment_rows])
train_split = concatenate_datasets([train_base, alignment_dataset]).shuffle(seed=44)

val_rows = []
for row in raw_dataset["validation"].shuffle(seed=45):
    if source_ok(dict(row)):
        val_rows.append(project_training_row(dict(row)))
    if len(val_rows) >= VALIDATION_SAMPLE_ROWS:
        break
validation_split = Dataset.from_list(val_rows)

dataset = DatasetDict({"train": train_split, "validation": validation_split})
print(dataset)
print("training rows:", len(dataset["train"]))
print("validation rows:", len(dataset["validation"]))
print("train tasks:", Counter(dataset["train"]["task_type"]))

## 6. Load Qwen3.5-9B With QLoRA

In [ ]:
import os, torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if SKIP_TRAINING:
    print("SKIP_TRAINING=True: model will load for smoke only, no Trainer run.")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True, token=os.environ["HF_TOKEN"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True,
    token=os.environ["HF_TOKEN"],
)
model.config.use_cache = bool(SKIP_TRAINING)

if not SKIP_TRAINING:
    # Qwen 9B + 4-bit LoRA still needs checkpointing on Colab/A100 when using
    # broad dataset coverage. This must be enabled both here and in Trainer args.
    prep_kwargs = {"use_gradient_checkpointing": USE_GRADIENT_CHECKPOINTING}
    if USE_GRADIENT_CHECKPOINTING:
        prep_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    model = prepare_model_for_kbit_training(model, **prep_kwargs)
    model.config.use_cache = False
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

candidate_targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
available_targets = sorted({name.split(".")[-1] for name, _ in model.named_modules() if name.split(".")[-1] in candidate_targets})
if not available_targets:
    raise RuntimeError("Could not find standard Qwen LoRA target modules. Inspect model.named_modules() before training.")
print("LoRA target modules:", available_targets)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=available_targets,
)
model = get_peft_model(model, lora_config)
if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()
model.eval() if SKIP_TRAINING else model.train()

## 7. Prompt Rendering And Tokenization

In [ ]:
SYSTEM_PROMPT = (
    "You are a Bangladesh contract, labor, company setup, and company policy vetting assistant. "
    "Answer from the supplied source context. Return valid JSON when the response expects JSON. "
    "Do not invent statutory thresholds, authority approvals, deadlines, remedies, or citations. "
    "Separate source-supported points from broader checks requiring other sources. "
    "This is exploration support, not legal advice."
)

def render_prompt(row):
    context = str(row.get("context", ""))[:MAX_CONTEXT_CHARS_FOR_TRAINING]
    return (
        f"<SYSTEM>\n{SYSTEM_PROMPT}\n</SYSTEM>\n"
        f"<INSTRUCTION>\n{row.get('instruction', '')}\n</INSTRUCTION>\n"
        f"<CONTEXT>\n{context}\n</CONTEXT>\n"
        "<RESPONSE>\n"
    )

def tokenize_row(row):
    prompt = render_prompt(row)
    response = str(row.get("response", "")).strip() + "\n</RESPONSE>"
    full = prompt + response + tokenizer.eos_token
    encoded = tokenizer(full, truncation=True, max_length=MAX_LEN, add_special_tokens=True)
    labels = encoded["input_ids"].copy()
    if RESPONSE_ONLY_TRAINING:
        prompt_encoded = tokenizer(prompt, truncation=True, max_length=MAX_LEN, add_special_tokens=True)
        prompt_len = min(len(prompt_encoded["input_ids"]), len(labels))
        labels[:prompt_len] = [-100] * prompt_len
    encoded["labels"] = labels
    encoded["length"] = len(encoded["input_ids"])
    return encoded

if SKIP_TRAINING:
    tokenized = None
    print("SKIP_TRAINING=True; tokenization skipped.")
else:
    tokenized = dataset.map(tokenize_row, remove_columns=dataset["train"].column_names, desc="tokenize qwen35 9b rows")
    print(tokenized)

## 8. Train With Wall-Clock Stop And Persistent Checkpoints

In [ ]:
import inspect, os, time, traceback, json
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from huggingface_hub import HfApi, create_repo, upload_folder

api = HfApi(token=os.environ["HF_TOKEN"])

def verify_adapter_dir(path):
    files = set(os.listdir(path)) if os.path.isdir(path) else set()
    assert "adapter_config.json" in files, f"adapter_config.json missing in {path}"
    assert ("adapter_model.safetensors" in files) or ("adapter_model.bin" in files), f"adapter weights missing in {path}"
    return files

def save_adapter_copy(model, target_dir, label):
    os.makedirs(target_dir, exist_ok=True)
    model.save_pretrained(target_dir, safe_serialization=True)
    tokenizer.save_pretrained(target_dir)
    files = verify_adapter_dir(target_dir)
    with open(os.path.join(target_dir, "persistence_manifest.json"), "w", encoding="utf-8") as f:
        json.dump({
            "label": label,
            "run_id": RUN_ID,
            "run_name": RUN_NAME,
            "base_model": BASE_MODEL,
            "dataset_repo": DATA_REPO,
            "adapter_repo": HF_OUTPUT_REPO,
            "saved_at": time.time(),
        }, f, indent=2)
    print(f"{label} saved:", target_dir, sorted(files))


def upload_adapter_copy(source_dir, path_in_repo, label):
    verify_adapter_dir(source_dir)
    create_repo(HF_OUTPUT_REPO, repo_type="model", private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
    for attempt in range(1, 4):
        try:
            upload_folder(
                repo_id=HF_OUTPUT_REPO,
                repo_type="model",
                folder_path=source_dir,
                path_in_repo=path_in_repo,
                token=os.environ["HF_TOKEN"],
                commit_message=f"{label} attempt {attempt}",
            )
            print(f"{label} uploaded to {path_in_repo}")
            return
        except Exception as exc:
            print(f"upload attempt {attempt} failed:", type(exc).__name__, exc)
            traceback.print_exc()
            time.sleep(10 * attempt)
    raise RuntimeError("all upload attempts failed for " + path_in_repo)

class AdapterPersistenceCallback(TrainerCallback):
    def on_save(self, args, state, control, model=None, **kwargs):
        if model is None or state.global_step <= 0:
            return control
        checkpoint_name = f"checkpoint-{state.global_step}"
        drive_checkpoint_dir = os.path.join(DRIVE_BACKUP_DIR, checkpoint_name)
        if BACKUP_TO_DRIVE:
            save_adapter_copy(model, drive_checkpoint_dir, f"Drive adapter checkpoint step {state.global_step}")
        if HUB_BACKUP_EVERY_SAVE:
            upload_adapter_copy(drive_checkpoint_dir, f"{HUB_CHECKPOINT_PREFIX}/{checkpoint_name}", f"Hub adapter checkpoint step {state.global_step}")
        return control

trainer = None
TRAINING_WAS_SKIPPED = bool(SKIP_TRAINING)

if SKIP_TRAINING:
    print("SKIP_TRAINING=True; skipping Trainer setup/training.")
else:
    resume_from = None
    if RESUME_FROM_DRIVE_CHECKPOINT and os.path.isdir(OUTPUT_DIR):
        latest_checkpoint = get_last_checkpoint(OUTPUT_DIR)
        if latest_checkpoint:
            resume_from = latest_checkpoint
            print("will resume from Drive Trainer checkpoint:", resume_from)

    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8, label_pad_token_id=-100, return_tensors="pt")
    effective_batch = max(1, BATCH_SIZE * GRAD_ACCUM)
    estimated_total_steps = MAX_STEPS if MAX_STEPS and MAX_STEPS > 0 else max(1, int((len(tokenized["train"]) * NUM_EPOCHS + effective_batch - 1) // effective_batch))
    EVAL_SAVE_STEPS = max(250, min(700, max(1, estimated_total_steps // 4)))
    WARMUP_STEPS = min(120, max(20, int(estimated_total_steps * 0.03)))
    print({"estimated_total_steps": estimated_total_steps, "max_steps": MAX_STEPS, "warmup_steps": WARMUP_STEPS, "eval_save_steps": EVAL_SAVE_STEPS})

    def make_training_arguments(**kwargs):
        """Create TrainingArguments across Colab Transformers versions.

        Some Colab runtimes expose older or patched Transformers builds where
        trainer keyword support differs. Filter by the installed signature
        instead of failing after paid setup work.
        """
        params = inspect.signature(TrainingArguments.__init__).parameters
        if "eval_strategy" not in params and "evaluation_strategy" in params and "eval_strategy" in kwargs:
            kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
        supported = {key: value for key, value in kwargs.items() if key in params and value is not None}
        dropped = sorted(set(kwargs) - set(supported))
        if dropped:
            print("TrainingArguments unsupported in this runtime; dropped:", dropped)
        return TrainingArguments(**supported)

    training_args = make_training_arguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False} if USE_GRADIENT_CHECKPOINTING else None,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=WARMUP_STEPS,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=EVAL_SAVE_STEPS,
        save_strategy="steps",
        save_steps=EVAL_SAVE_STEPS,
        save_total_limit=2,
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        report_to="tensorboard",
        run_name=RUN_NAME,
        remove_unused_columns=False,
        dataloader_num_workers=1,
    )

    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=collator,
        callbacks=[AdapterPersistenceCallback()],
    )
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer
    trainer = Trainer(**trainer_kwargs)
    trainer.train(resume_from_checkpoint=resume_from)
    TRAINING_WAS_SKIPPED = False

## 9. Final Save / Push

In [ ]:
import os, json, time
from huggingface_hub import upload_file

if SKIP_TRAINING or trainer is None:
    print("Training skipped; final save skipped.")
else:
    metrics = trainer.evaluate()
    metrics.update({
        "base_model": BASE_MODEL,
        "dataset_repo": DATA_REPO,
        "adapter_repo": HF_OUTPUT_REPO,
        "run_name": RUN_NAME,
        "max_steps": MAX_STEPS,
        "finished_at": time.time(),
    })
    print(metrics)

    save_adapter_copy(trainer.model, FINAL_ADAPTER_DIR, "Final local adapter")
    if BACKUP_TO_DRIVE:
        save_adapter_copy(trainer.model, DRIVE_FINAL_ADAPTER_DIR, "Final Drive adapter")
    create_repo(HF_OUTPUT_REPO, repo_type="model", private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
    trainer.model.push_to_hub(HF_OUTPUT_REPO, private=True, token=os.environ["HF_TOKEN"], commit_message="Upload Qwen3.5 9B max-coverage LoRA")
    tokenizer.push_to_hub(HF_OUTPUT_REPO, private=True, token=os.environ["HF_TOKEN"], commit_message="Upload tokenizer")
    with open("/content/qwen35_9b_3h_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    upload_file(
        path_or_fileobj="/content/qwen35_9b_3h_metrics.json",
        path_in_repo="training_metrics.json",
        repo_id=HF_OUTPUT_REPO,
        repo_type="model",
        token=os.environ["HF_TOKEN"],
        commit_message="Upload training metrics",
    )
    print("Pushed final adapter to", HF_OUTPUT_REPO)

## 10. Quick Smoke Test

In [ ]:
import json, torch
from transformers import GenerationConfig


def first_balanced_json(text):
    start = text.find("{")
    if start < 0:
        return text.strip()
    depth = 0
    in_string = False
    escape = False
    for idx, ch in enumerate(text[start:], start=start):
        if escape:
            escape = False
            continue
        if ch == "\\":
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:idx + 1].strip()
    return text[start:].strip()

def clean_generation(text):
    for marker in ("</RESPONSE>", "<SYSTEM>", "<INSTRUCTION>", "<CONTEXT>"):
        if marker in text:
            text = text.split(marker, 1)[0]
    return first_balanced_json(text.strip())

def deterministic_generation_config(max_new_tokens=900):
    config = GenerationConfig.from_model_config(model.config)
    config.do_sample = False
    config.num_beams = 1
    config.repetition_penalty = 1.03
    config.max_new_tokens = max_new_tokens
    config.pad_token_id = tokenizer.pad_token_id
    config.eos_token_id = tokenizer.eos_token_id
    for attr in ("temperature", "top_p", "top_k", "min_p", "typical_p"):
        if hasattr(config, attr):
            setattr(config, attr, None)
    return config

def generate_for_row(row, max_new_tokens=900):
    prompt = render_prompt(row)
    prefix = "{\n"
    device = next(model.parameters()).device
    inputs = tokenizer(prompt + prefix, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        output = model.generate(**inputs, generation_config=deterministic_generation_config(max_new_tokens=max_new_tokens))
    raw = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return clean_generation(prefix + raw)

model.eval()
probe_rows = [company_anchor, alignment_rows[-1]]
score = {"valid_json": 0, "has_citations_or_refusal": 0, "total": len(probe_rows)}
for idx, probe in enumerate(probe_rows, start=1):
    generated = generate_for_row(probe)
    try:
        parsed = json.loads(generated)
        score["valid_json"] += 1
    except Exception:
        parsed = {"raw": generated[:1000]}
    lowered = json.dumps(parsed, ensure_ascii=False).lower()
    score["has_citations_or_refusal"] += int(bool(parsed.get("citations")) or "cannot" in lowered)
    print("\n=== probe", idx, probe.get("task_type"), "===")
    print(generated[:2400])
print("\nsmoke_score", score)